# Extract JSON-LD from Common Crawl

This notebook extracts JSON-LD semantic annotations from Common Crawl for a specific domain.
It queries the **Parquet columnar index** directly from S3, then fetches individual WARC records via HTTP range requests.

In [1]:
!pip install duckdb requests beautifulsoup4 warcio

Defaulting to user installation because normal site-packages is not writeable


In [2]:
!pip install --upgrade ipywidgets widgetsnbextension

Defaulting to user installation because normal site-packages is not writeable


In [3]:
import io
import json
import time

import duckdb
import requests
from bs4 import BeautifulSoup
from warcio.archiveiterator import ArchiveIterator

CC_DATA_BASE = "https://data.commoncrawl.org"

## Configuration

- `DOMAIN`: the registered domain (e.g. `lemonde.fr`, `reuters.com`)
- `URL_PREFIX`: optional path prefix to filter by (e.g. `https://www.lemonde.fr/en/`). Set to `None` to get all pages from the domain.
- `CRAWL_ID`: the Common Crawl crawl identifier. Check https://commoncrawl.org/overview for available crawls.

In [4]:
DOMAIN = "lemonde.fr"                          # registered domain
URL_PREFIX = "https://www.lemonde.fr/en/"       # set to None to skip path filtering
LIMIT = 100                                     # max pages to fetch
CRAWL_ID = "CC-MAIN-2026-12"                    # latest crawl as of April 2026
OUTPUT_FILE = "jsonld_results.jsonl"

## Step 1: Check which paths file is available

Common Crawl publishes two index listings per crawl:
- `cc-index.paths.gz` → CDX text files (NOT Parquet)
- `cc-index-table.paths.gz` → Parquet files (what we need)

If neither is available, we fall back to querying the global Parquet table on S3 directly.

In [5]:
for name in ["cc-index.paths.gz", "cc-index-table.paths.gz"]:
    url = f"{CC_DATA_BASE}/crawl-data/{CRAWL_ID}/{name}"
    resp = requests.head(url, timeout=30)
    print(f"{name}: HTTP {resp.status_code}")

cc-index.paths.gz: HTTP 200
cc-index-table.paths.gz: HTTP 200


## Step 2: Query the Parquet columnar index

We query the Parquet table directly from S3 using the partition path:
`s3://commoncrawl/cc-index/table/cc-main/warc/crawl={CRAWL_ID}/subset=warc/*.parquet`

This avoids needing any paths listing file. DuckDB reads the Parquet files via S3 (no AWS credentials needed — it's public data, but must use `us-east-1` region).

**Note:** This query can take several minutes as it scans large Parquet files remotely.

In [6]:
import gzip

paths_url = f"{CC_DATA_BASE}/crawl-data/{CRAWL_ID}/cc-index-table.paths.gz"
resp = requests.get(paths_url, timeout=60)
resp.raise_for_status()

paths = gzip.decompress(resp.content).decode().splitlines()

parquet_files = [
    f"https://data.commoncrawl.org/{p}"
    for p in paths
    if p.endswith(".parquet")
]

print(f"Found {len(parquet_files)} parquet files")

Found 900 parquet files


In [ ]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
# con.execute("PRAGMA enable_progress_bar;")
con.execute("PRAGMA disable_progress_bar;") 
records = []

# DOMAIN = "wikipedia.org"
# LIMIT = 50

DOMAINS = ["en.wikipedia.org"]
domains_sql = ", ".join(f"'{d}'" for d in DOMAINS)


for i, file in enumerate(parquet_files):
    try:
        if (i+1) % 50 == 0:
            print(f"[{i+1}/{len(parquet_files)}] Processing: {file}")

        rows = con.execute(f"""
            SELECT
                url,
                warc_filename,
                warc_record_offset,
                warc_record_length,
                url_host_name,
                url_host_registered_domain,
                fetch_time,
                content_languages,
                content_mime_detected,
                fetch_status
            FROM read_parquet('{file}')
            WHERE url_host_name IN ({domains_sql})
            AND fetch_status = 200
            AND content_mime_detected = 'text/html'
        """).fetchall()

        # rows = con.execute(f"""
        #     SELECT url, warc_filename, warc_record_offset, warc_record_length
        #     FROM read_parquet('{file}')
        #     WHERE url_host_registered_domain = '{DOMAIN}'
        #     LIMIT {LIMIT}
        # """).fetchall()

        for r in rows:
            records.append({
                "url": r[0],
                "warc_filename": r[1],
                "warc_record_offset": r[2],
                "warc_record_length": r[3],
                "url_host_name": r[4],
                "url_host_registered_domain": r[5],
                "fetch_time": r[6],
                "content_languages": r[7],
                "content_mime_detected": r[8],
                "fetch_status": r[9]
            })

        # if len(records) >= LIMIT:
        #     print("Reached LIMIT, stopping early.")
        #     break

    except Exception as e:
        print(f"Skipping broken file: {file}")
        continue

con.close()

print(f"Total records found: {len(records)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[50/900] Processing: https://data.commoncrawl.org/cc-index/table/cc-main/warc/crawl=CC-MAIN-2026-12/subset=crawldiagnostics/part-00049-34e6fda8-4578-4144-973f-2adf16729186.c000.gz.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[100/900] Processing: https://data.commoncrawl.org/cc-index/table/cc-main/warc/crawl=CC-MAIN-2026-12/subset=crawldiagnostics/part-00099-34e6fda8-4578-4144-973f-2adf16729186.c000.gz.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (2 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Skipping broken file: https://data.commoncrawl.org/cc-index/table/cc-main/warc/crawl=CC-MAIN-2026-12/subset=crawldiagnostics/part-00196-34e6fda8-4578-4144-973f-2adf16729186.c000.gz.parquet
[200/900] Processing: https://data.commoncrawl.org/cc-index/table/cc-main/warc/crawl=CC-MAIN-2026-12/subset=crawldiagnostics/part-00199-34e6fda8-4578-4144-973f-2adf16729186.c000.gz.parquet


In [ ]:
# Save output to csv 
import pandas as pd

df = pd.DataFrame(records)
df.to_csv("records_wiki.csv", index=False)

### Quick look at what we found

In [11]:
records[:5]

[{'url': 'https://en.wikipedia.org/?curid=13729',
  'warc_filename': 'crawl-data/CC-MAIN-2026-12/segments/1772687277647.32/warc/CC-MAIN-20260308165213-20260308195213-00802.warc.gz',
  'warc_record_offset': 156364261,
  'warc_record_length': 139583,
  'url_host_name': 'en.wikipedia.org',
  'url_host_registered_domain': 'wikipedia.org',
  'fetch_time': datetime.datetime(2026, 3, 8, 18, 31, 26, tzinfo=<DstTzInfo 'Europe/Paris' CET+1:00:00 STD>),
  'content_languages': 'eng,dan',
  'content_mime_detected': 'text/html',
  'fetch_status': 200},
 {'url': 'https://en.wikipedia.org/?curid=1504834',
  'warc_filename': 'crawl-data/CC-MAIN-2026-12/segments/1772687553247.70/warc/CC-MAIN-20260316014608-20260316044608-00215.warc.gz',
  'warc_record_offset': 172446922,
  'warc_record_length': 28644,
  'url_host_name': 'en.wikipedia.org',
  'url_host_registered_domain': 'wikipedia.org',
  'fetch_time': datetime.datetime(2026, 3, 16, 5, 5, 44, tzinfo=<DstTzInfo 'Europe/Paris' CET+1:00:00 STD>),
  'conte

In [13]:
len(records)

102422

In [12]:
unique_domains = set(r["url_host_registered_domain"] for r in records)
print(unique_domains)

{'wikipedia.org'}


In [12]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
# con.execute("PRAGMA enable_progress_bar;")
con.execute("PRAGMA disable_progress_bar;") 

records = []

# DOMAIN = "wikipedia.org"
# LIMIT = 100

DOMAINS = ["bbc.com", "bbc.co.uk", "france24.com", "lemonde.fr", "cnn.com", "aljazeera.com", "thetimes.com", "politifact.com"]

# DOMAINS = ["bbc.com", "bbc.co.uk"]

domains_sql = ", ".join(f"'{d}'" for d in DOMAINS)


for i, file in enumerate(parquet_files):
    try:
        if (i+1) % 50 == 0:
            print(f"[{i+1}/{len(parquet_files)}] Processing: {file}")

        rows = con.execute(f"""
            SELECT
                url,
                warc_filename,
                warc_record_offset,
                warc_record_length,
                url_host_name,
                url_host_registered_domain,
                fetch_time,
                content_languages,
                content_mime_detected,
                fetch_status
            FROM read_parquet('{file}')
            WHERE url_host_registered_domain IN ({domains_sql})
        """).fetchall()

        # rows = con.execute(f"""
        #     SELECT url, warc_filename, warc_record_offset, warc_record_length
        #     FROM read_parquet('{file}')
        #     WHERE url_host_registered_domain = '{DOMAIN}'
        #     LIMIT {LIMIT}
        # """).fetchall()

        for r in rows:
            records.append({
                "url": r[0],
                "warc_filename": r[1],
                "warc_record_offset": r[2],
                "warc_record_length": r[3],
                "url_host_name": r[4],
                "url_host_registered_domain": r[5],
                "fetch_time": r[6],
                "content_languages": r[7],
                "content_mime_detected": r[8],
                "fetch_status": r[9]
            })

        # if len(records) >= LIMIT:
        #     print("Reached LIMIT, stopping early.")
        #     break

    except Exception as e:
        print(f"Skipping broken file: {file}")
        continue

con.close()

print(f"Total records found: {len(records)}")

[50/900] Processing: https://data.commoncrawl.org/cc-index/table/cc-main/warc/crawl=CC-MAIN-2026-12/subset=crawldiagnostics/part-00049-34e6fda8-4578-4144-973f-2adf16729186.c000.gz.parquet
[100/900] Processing: https://data.commoncrawl.org/cc-index/table/cc-main/warc/crawl=CC-MAIN-2026-12/subset=crawldiagnostics/part-00099-34e6fda8-4578-4144-973f-2adf16729186.c000.gz.parquet
[150/900] Processing: https://data.commoncrawl.org/cc-index/table/cc-main/warc/crawl=CC-MAIN-2026-12/subset=crawldiagnostics/part-00149-34e6fda8-4578-4144-973f-2adf16729186.c000.gz.parquet
[200/900] Processing: https://data.commoncrawl.org/cc-index/table/cc-main/warc/crawl=CC-MAIN-2026-12/subset=crawldiagnostics/part-00199-34e6fda8-4578-4144-973f-2adf16729186.c000.gz.parquet
[250/900] Processing: https://data.commoncrawl.org/cc-index/table/cc-main/warc/crawl=CC-MAIN-2026-12/subset=crawldiagnostics/part-00249-34e6fda8-4578-4144-973f-2adf16729186.c000.gz.parquet
[300/900] Processing: https://data.commoncrawl.org/cc-in

In [13]:
# Save output to csv 
import pandas as pd

df = pd.DataFrame(records)
df.to_csv("records_other.csv", index=False)

In [14]:
len(records)

124950

In [15]:
import pandas as pd

# ── 1. Shape & basic info ──────────────────────────────────────────
print(f"Total records: {len(df)}")
print(f"Columns: {list(df.columns)}\n")

# ── 2. Records per domain ──────────────────────────────────────────
print("=== Records per domain ===")
print(df["url_host_registered_domain"].value_counts().to_string(), "\n")

# ── 3. Fetch status per domain ─────────────────────────────────────
print("=== Fetch status per domain ===")
print(df.groupby(["url_host_registered_domain", "fetch_status"]).size().to_string(), "\n")

# ── 4. WARC subset breakdown (warc / crawldiagnostics / robotstxt) ─
df["warc_subset"] = df["warc_filename"].str.extract(r"subset=(\w+)/")
print("=== WARC subset per domain ===")
print(df.groupby(["url_host_registered_domain", "warc_subset"]).size().to_string(), "\n")

# ── 5. MIME type distribution ──────────────────────────────────────
print("=== MIME types ===")
print(df["content_mime_detected"].value_counts().to_string(), "\n")

# ── 6. Language distribution (non-null only) ───────────────────────
print("=== Content languages (top 10) ===")
print(df["content_languages"].dropna().value_counts().head(10).to_string(), "\n")

# ── 7. WARC record size stats ──────────────────────────────────────
print("=== WARC record size (bytes) ===")
print(df["warc_record_length"].describe().to_string(), "\n")

# ── 8. Fetch time range ────────────────────────────────────────────
print("=== Fetch time range ===")
print(f"  Earliest: {df['fetch_time'].min()}")
print(f"  Latest:   {df['fetch_time'].max()}")


Total records: 124950
Columns: ['url', 'warc_filename', 'warc_record_offset', 'warc_record_length', 'url_host_name', 'url_host_registered_domain', 'fetch_time', 'content_languages', 'content_mime_detected', 'fetch_status']

=== Records per domain ===
url_host_registered_domain
aljazeera.com     73704
thetimes.com      29625
politifact.com    13131
cnn.com            8184
france24.com        301
bbc.com               3
lemonde.fr            1
bbc.co.uk             1 

=== Fetch status per domain ===
url_host_registered_domain  fetch_status
aljazeera.com               200             62693
                            301             10648
                            302                31
                            304                 4
                            400                 1
                            403                 4
                            404               321
                            406                 1
                            500                 1
bbc.c

In [23]:
df[df["url_host_registered_domain"] == "bbc.com"]

,url,warc_filename,warc_record_offset,warc_record_length,url_host_name,url_host_registered_domain,fetch_time,content_languages,content_mime_detected,fetch_status,warc_subset
29109,https://bbc.com/robots.txt,crawl-data/CC-MAIN-2026-12/segments/1772687277...,507487,908,bbc.com,bbc.com,2026-03-07 16:34:49+01:00,None,text/plain,301,NaN
29110,https://www.bbc.com/robots.txt,crawl-data/CC-MAIN-2026-12/segments/1772687277...,1266543,2513,www.bbc.com,bbc.com,2026-03-07 16:34:49+01:00,None,text/plain,200,NaN
29111,https://player.bbc.com/robots.txt,crawl-data/CC-MAIN-2026-12/segments/1772687277...,1154162,955,player.bbc.com,bbc.com,2026-03-10 13:08:35+01:00,None,text/x-robots,200,NaN


In [16]:
df_200 = df[df["fetch_status"] == 200]
df_200.to_csv("records_other_200.csv", index=False)
print(f"Records with 200: {len(df_200)}")


Records with 200: 95758


In [17]:
df_200_html = df[(df["fetch_status"] == 200) & (df["content_mime_detected"] == "text/html")]
df_200_html.to_csv("records_200_html.csv", index=False)
print(f"Records with 200 + text/html: {len(df_200_html)}")

Records with 200 + text/html: 94959


In [19]:
df_cnn = df_200_html[df_200_html["url_host_registered_domain"] == "cnn.com"]
df_cnn.to_csv("records_cnn_200_html.csv", index=False)
print(f"CNN Records with 200 + text/html: {len(df_cnn)}")


CNN Records with 200 + text/html: 3277


In [20]:
df_fr24 = df_200_html[df_200_html["url_host_registered_domain"] == "france24.com"]
df_fr24.to_csv("records_fr24_200_html.csv", index=False)
print(f"FR24 Records with 200 + text/html: {len(df_fr24)}")

FR24 Records with 200 + text/html: 165


## Step 3 & 4: Fetch WARC records and extract JSON-LD

For each page found in the index, we:
1. Send an HTTP range request to download just that one WARC record (~50-200KB)
2. Parse the HTML and extract `<script type="application/ld+json">` blocks

In [ ]:
def fetch_warc_record(filename, offset, length):
    """Fetch a single WARC record via HTTP range request, return HTML or None."""
    url = f"{CC_DATA_BASE}/{filename}"
    headers = {"Range": f"bytes={offset}-{offset + length - 1}"}
    try:
        resp = requests.get(url, headers=headers, timeout=60)
        resp.raise_for_status()
    except requests.RequestException:
        return None

    raw = io.BytesIO(resp.content)
    try:
        for rec in ArchiveIterator(raw):
            if rec.rec_type == "response":
                html_bytes = rec.content_stream().read()
                try:
                    return html_bytes.decode("utf-8")
                except UnicodeDecodeError:
                    return html_bytes.decode("latin-1")
    except Exception:
        return None
    return None


def extract_jsonld(html):
    """Extract all JSON-LD blocks from HTML."""
    soup = BeautifulSoup(html, "html.parser")
    results = []
    for script in soup.find_all("script", attrs={"type": "application/ld+json"}):
        text = script.string
        if not text:
            continue
        try:
            data = json.loads(text)
            if isinstance(data, dict):
                results.append(data)
            elif isinstance(data, list):
                results.extend(data)
        except json.JSONDecodeError:
            pass
    return results

In [ ]:
stats = {"fetched": 0, "with_jsonld": 0, "total_blocks": 0}
all_results = []

with open(OUTPUT_FILE, "w") as f_out:
    for i, rec in enumerate(records):
        url = rec["url"]
        print(f"[{i+1}/{len(records)}] {url}")

        html = fetch_warc_record(
            rec["warc_filename"], rec["warc_record_offset"], rec["warc_record_length"]
        )
        if html is None:
            print("  x fetch failed")
            continue
        stats["fetched"] += 1

        jsonld_blocks = extract_jsonld(html)
        if jsonld_blocks:
            stats["with_jsonld"] += 1
            stats["total_blocks"] += len(jsonld_blocks)

            for block in jsonld_blocks:
                entry = {"url": url, "domain": DOMAIN, "jsonld": block}
                all_results.append(entry)
                f_out.write(json.dumps(entry, ensure_ascii=False) + "\n")

            types = [b.get("@type", "unknown") for b in jsonld_blocks]
            print(f"  OK {len(jsonld_blocks)} block(s): {types}")
        else:
            print("  - no JSON-LD")

        time.sleep(0.3)

print(f"\nDone! {stats}")

## Summary & Quick Analysis

In [ ]:
print(f"Crawl:               {CRAWL_ID}")
print(f"Domain:              {DOMAIN}")
print(f"URL prefix:          {URL_PREFIX or '(all)'}")
print(f"Pages in index:      {len(records)}")
print(f"Pages fetched:       {stats['fetched']}")
print(f"Pages with JSON-LD:  {stats['with_jsonld']}")
print(f"Total JSON-LD blocks:{stats['total_blocks']}")
print(f"Output file:         {OUTPUT_FILE}")

if stats["fetched"] > 0:
    pct = stats["with_jsonld"] / stats["fetched"] * 100
    print(f"\n% of pages with JSON-LD: {pct:.1f}%")

In [ ]:
from collections import Counter

type_counter = Counter()
for entry in all_results:
    t = entry["jsonld"].get("@type", "unknown")
    if isinstance(t, list):
        for x in t:
            type_counter[x] += 1
    else:
        type_counter[t] += 1

print("@type distribution:")
for typ, count in type_counter.most_common(20):
    print(f"  {typ}: {count}")

In [ ]:
prop_counter = Counter()
for entry in all_results:
    for key in entry["jsonld"].keys():
        if not key.startswith("@"):
            prop_counter[key] += 1

print("Top 30 properties used:")
for prop, count in prop_counter.most_common(30):
    print(f"  {prop}: {count}")

In [ ]:
if all_results:
    print(json.dumps(all_results[0]["jsonld"], indent=2, ensure_ascii=False)[:3000])